# Daily Challenge — Multi-Attention & Transformer Comparisons

**Course:** Developers Institute  **Week 7 - Day 2**  
**Author:** Alex Goldbaum

Build attention from scratch in PyTorch (single-head → multi-head → encoder),
train it on the **SNLI** natural language inference dataset, compare against
a pretrained **DistilBERT** baseline fine-tuned on the same data, and
visualize the resulting attention maps to interpret what each model focuses on.

**⚠️ Use a GPU runtime in Colab** (Runtime → Change runtime type → GPU).
Training the custom encoder + fine-tuning DistilBERT on CPU is impractical.


## Setup


In [ ]:
%pip install -qU torch transformers==4.* datasets==2.* einops scikit-learn matplotlib seaborn


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import math, random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 1. Scaled Dot-Product Attention (Single Head)

We implement the canonical scaled dot-product attention from the paper:

$$ \text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V $$

The block takes a `(batch, seq_len, hidden_dim)` tensor, projects it into
Q, K, V with three `nn.Linear` layers, scales the scores by `1/sqrt(d_k)` for
numerical stability, and returns both the attended output and the attention
weights for inspection.


In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.scale = 1.0 / math.sqrt(hidden_dim)
        self.W_q = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_k = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_v = nn.Linear(hidden_dim, hidden_dim, bias=False)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None):
        # x: (batch, seq_len, hidden_dim)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        return out, attn


# Shape validation with a dummy tensor
B, L, D = 4, 12, 64  # batch, seq_len, hidden_dim
x_dummy = torch.randn(B, L, D)
sha = SingleHeadAttention(hidden_dim=D)
out, attn = sha(x_dummy)
print(f'Input shape : {x_dummy.shape}')
print(f'Output shape: {out.shape}')
print(f'Attention   : {attn.shape}  (batch, seq_len, seq_len)')
print(f'Attention row sum (should be ~1): {attn[0, 0].sum().item():.4f}')


In [ ]:
# Log the attention weights for one sample
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(attn[0].detach().numpy(), cmap='viridis', cbar=True, ax=ax)
ax.set_title('Single-head attention — random init, dummy input', fontweight='bold')
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
plt.tight_layout()
plt.show()


## 2. Multi-Head Attention

Multi-head attention runs `num_heads` parallel scaled dot-product attentions
on slices of the embedding dimension, then concatenates and projects back to
`hidden_dim`. We add **dropout** on the attention weights and on the output,
plus a **residual connection** so the block can be stacked.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0, 'hidden_dim must be divisible by num_heads'
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)

        self.W_q = nn.Linear(hidden_dim, hidden_dim)
        self.W_k = nn.Linear(hidden_dim, hidden_dim)
        self.W_v = nn.Linear(hidden_dim, hidden_dim)
        self.W_o = nn.Linear(hidden_dim, hidden_dim)

        self.attn_dropout = nn.Dropout(dropout)
        self.out_dropout = nn.Dropout(dropout)

    def _split_heads(self, t: torch.Tensor) -> torch.Tensor:
        # (B, L, D) -> (B, H, L, head_dim)
        B, L, _ = t.shape
        t = t.reshape(B, L, self.num_heads, self.head_dim)
        return t.transpose(1, 2)

    def _merge_heads(self, t: torch.Tensor) -> torch.Tensor:
        # (B, H, L, head_dim) -> (B, L, D)
        t = t.transpose(1, 2).contiguous()
        B, L, H, hd = t.shape
        return t.reshape(B, L, H * hd)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None):
        # x: (B, L, D), mask: (B, L) - 1 for tokens, 0 for pad
        residual = x

        Q = self._split_heads(self.W_q(x))  # (B, H, L, hd)
        K = self._split_heads(self.W_k(x))
        V = self._split_heads(self.W_v(x))

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale  # (B, H, L, L)
        if mask is not None:
            # (B, L) -> (B, 1, 1, L) broadcastable to (B, H, L, L)
            m = mask.unsqueeze(1).unsqueeze(1)
            scores = scores.masked_fill(m == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)

        ctx = torch.matmul(attn, V)         # (B, H, L, hd)
        ctx = self._merge_heads(ctx)        # (B, L, D)
        ctx = self.out_dropout(self.W_o(ctx))
        return ctx + residual, attn         # residual connection


# Forward example
B, L, D, H = 4, 16, 128, 8
x = torch.randn(B, L, D)
mha = MultiHeadAttention(hidden_dim=D, num_heads=H, dropout=0.1)
y, attn = mha(x)
print(f'Input  shape: {x.shape}')
print(f'Output shape: {y.shape}')
print(f'Attention   : {attn.shape}  (batch, num_heads, seq_len, seq_len)')


## 3. Transformer Encoder Block

A standard encoder block stacks `MultiHeadAttention → LayerNorm → FFN →
LayerNorm`. The feed-forward sub-layer is the typical 2-layer MLP with GELU.
Both sub-layers use a residual connection (handled inside `MultiHeadAttention`
above and explicitly here in the FFN).


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, hidden_dim: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, hidden_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x) + x  # residual


class EncoderBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(hidden_dim, num_heads, dropout)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.ffn = FeedForward(hidden_dim, ff_dim, dropout)
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x, mask=None):
        x, attn = self.attn(x, mask)
        x = self.norm1(x)
        x = self.ffn(x)
        x = self.norm2(x)
        return x, attn


# Sanity check
enc = EncoderBlock(hidden_dim=128, num_heads=8, ff_dim=256)
y, attn = enc(torch.randn(2, 16, 128))
print(f'Encoder output: {y.shape}  | attn: {attn.shape}')


## 4. (Optional) Custom Encoder Trained on SNLI

We tokenize the premise + hypothesis with the DistilBERT tokenizer (so the
two models share an input vocabulary for a fair comparison), wrap the
encoder stack with an embedding + classification head, and train for a
couple of epochs on a subset of SNLI. SNLI has 3 classes: `entailment` (0),
`neutral` (1), `contradiction` (2).


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

snli = load_dataset('snli')
# SNLI has -1 labels for examples without consensus annotations — drop them
snli = snli.filter(lambda ex: ex['label'] != -1)

# Subset for tractability in Colab (still enough signal to learn)
TRAIN_SAMPLES = 20_000
VAL_SAMPLES = 2_000
TEST_SAMPLES = 2_000

snli['train'] = snli['train'].shuffle(seed=RANDOM_STATE).select(range(TRAIN_SAMPLES))
snli['validation'] = snli['validation'].shuffle(seed=RANDOM_STATE).select(range(VAL_SAMPLES))
snli['test'] = snli['test'].shuffle(seed=RANDOM_STATE).select(range(TEST_SAMPLES))

label_names = ['entailment', 'neutral', 'contradiction']
print({split: len(snli[split]) for split in snli})
print('Example:', snli['train'][0])


In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 96
BATCH_SIZE = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID = tokenizer.pad_token_id


def tokenize(batch):
    return tokenizer(
        batch['premise'], batch['hypothesis'],
        truncation=True, padding='max_length', max_length=MAX_LEN,
    )


tokenized = snli.map(tokenize, batched=True)
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

train_loader = DataLoader(tokenized['train'], batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(tokenized['validation'], batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(tokenized['test'], batch_size=BATCH_SIZE, shuffle=False)
print('Loaders ready.')


In [ ]:
class CustomEncoderClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes=3, hidden_dim=128, num_heads=8,
                 num_layers=2, ff_dim=256, max_len=128, dropout=0.1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.token_emb = nn.Embedding(vocab_size, hidden_dim, padding_idx=pad_id)
        self.pos_emb = nn.Embedding(max_len, hidden_dim)
        self.input_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            EncoderBlock(hidden_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        B, L = input_ids.shape
        positions = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        x = self.input_dropout(x)
        attentions = []
        for layer in self.layers:
            x, attn = layer(x, mask=attention_mask)
            attentions.append(attn)
        # Masked mean pooling over non-pad tokens
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        logits = self.classifier(pooled)
        return logits, attentions


custom_model = CustomEncoderClassifier(
    vocab_size=VOCAB_SIZE,
    num_classes=3,
    hidden_dim=128,
    num_heads=8,
    num_layers=2,
    ff_dim=256,
    max_len=MAX_LEN,
    dropout=0.1,
    pad_id=PAD_ID,
).to(device)

n_params = sum(p.numel() for p in custom_model.parameters() if p.requires_grad)
print(f'Custom encoder trainable params: {n_params:,}')


In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            logits, _ = model(input_ids, attention_mask)
            pred = logits.argmax(dim=-1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    return correct / total


EPOCHS = 3
LR = 5e-4
optimizer = torch.optim.AdamW(custom_model.parameters(), lr=LR, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

history = {'train_loss': [], 'val_acc': []}
for epoch in range(1, EPOCHS + 1):
    custom_model.train()
    epoch_loss, n = 0.0, 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        logits, _ = custom_model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(custom_model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item() * input_ids.size(0)
        n += input_ids.size(0)
    train_loss = epoch_loss / n
    val_acc = evaluate(custom_model, val_loader)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch}: train_loss={train_loss:.4f} | val_acc={val_acc:.4f}')

custom_test_acc = evaluate(custom_model, test_loader)
print(f'\nCustom encoder test accuracy: {custom_test_acc:.4f}')


## 5. Pretrained Baseline — DistilBERT Fine-Tuned on SNLI

Same data, same tokenizer, same labels. The point is the contrast: how much
does pre-training buy us, with the same compute spent on fine-tuning?


In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

id2label = {i: n for i, n in enumerate(label_names)}
label2id = {n: i for i, n in enumerate(label_names)}

pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id,
).to(device)

optimizer_pt = torch.optim.AdamW(pretrained_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer_pt, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps,
)


def evaluate_hf(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            pred = out.logits.argmax(dim=-1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    return correct / total


history_pt = {'train_loss': [], 'val_acc': []}
for epoch in range(1, EPOCHS + 1):
    pretrained_model.train()
    epoch_loss, n = 0.0, 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer_pt.zero_grad()
        out = pretrained_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(pretrained_model.parameters(), 1.0)
        optimizer_pt.step()
        scheduler.step()
        epoch_loss += out.loss.item() * input_ids.size(0)
        n += input_ids.size(0)
    train_loss = epoch_loss / n
    val_acc = evaluate_hf(pretrained_model, val_loader)
    history_pt['train_loss'].append(train_loss)
    history_pt['val_acc'].append(val_acc)
    print(f'Epoch {epoch}: train_loss={train_loss:.4f} | val_acc={val_acc:.4f}')

pretrained_test_acc = evaluate_hf(pretrained_model, test_loader)
print(f'\nDistilBERT test accuracy: {pretrained_test_acc:.4f}')


In [ ]:
# Side-by-side comparison
import pandas as pd

comparison = pd.DataFrame({
    'Model': ['Custom encoder (2 layers, from scratch)', 'DistilBERT (fine-tuned)'],
    'Trainable params': [n_params, sum(p.numel() for p in pretrained_model.parameters())],
    'Test accuracy': [custom_test_acc, pretrained_test_acc],
})
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history['val_acc'], 'o-', label='Custom encoder')
axes[0].plot(history_pt['val_acc'], 's-', label='DistilBERT')
axes[0].set_title('Validation accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()

axes[1].plot(history['train_loss'], 'o-', label='Custom encoder')
axes[1].plot(history_pt['train_loss'], 's-', label='DistilBERT')
axes[1].set_title('Training loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Cross-entropy'); axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Attention Map Visualization

Pick two sample sentence pairs from the validation set. For each, plot the
averaged head attention from the **last layer** of the custom encoder and
the same view from DistilBERT, so we can compare what each model focuses on.


In [ ]:
def get_pretrained_attention(text_a, text_b):
    enc = tokenizer(text_a, text_b, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
    out = pretrained_model.distilbert(**enc, output_attentions=True)
    # last layer attentions: (1, heads, L, L)
    return out.attentions[-1][0].mean(dim=0).detach().cpu().numpy(), enc


def get_custom_attention(text_a, text_b):
    enc = tokenizer(text_a, text_b, return_tensors='pt', truncation=True,
                    padding='max_length', max_length=MAX_LEN).to(device)
    custom_model.eval()
    with torch.no_grad():
        _, attentions = custom_model(enc['input_ids'], enc['attention_mask'])
    return attentions[-1][0].mean(dim=0).detach().cpu().numpy(), enc


def truncate_to_real_tokens(attn_2d, attention_mask):
    n = int(attention_mask.sum().item())
    return attn_2d[:n, :n], n


samples = [
    ('A man is playing the guitar on stage.',
     'A musician is performing in front of an audience.',
     'entailment'),
    ('The cat is sleeping on the couch.',
     'The dog is running in the park.',
     'contradiction'),
]

for premise, hypothesis, true_label in samples:
    print(f'\n{premise}  +  {hypothesis}  ({true_label})')

    attn_p, enc_p = get_pretrained_attention(premise, hypothesis)
    attn_c, enc_c = get_custom_attention(premise, hypothesis)

    attn_p_t, n_p = truncate_to_real_tokens(attn_p, enc_p['attention_mask'][0])
    attn_c_t, n_c = truncate_to_real_tokens(attn_c, enc_c['attention_mask'][0])
    tokens_p = tokenizer.convert_ids_to_tokens(enc_p['input_ids'][0][:n_p])
    tokens_c = tokenizer.convert_ids_to_tokens(enc_c['input_ids'][0][:n_c])

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    sns.heatmap(attn_c_t, xticklabels=tokens_c, yticklabels=tokens_c,
                cmap='viridis', ax=axes[0])
    axes[0].set_title('Custom encoder — last layer (avg heads)', fontweight='bold')
    axes[0].tick_params(axis='x', labelrotation=70)

    sns.heatmap(attn_p_t, xticklabels=tokens_p, yticklabels=tokens_p,
                cmap='viridis', ax=axes[1])
    axes[1].set_title('DistilBERT — last layer (avg heads)', fontweight='bold')
    axes[1].tick_params(axis='x', labelrotation=70)
    plt.tight_layout()
    plt.show()


## 7. Reflection — Custom Encoder vs Pretrained DistilBERT

**Where each model lands on test accuracy.** The pretrained DistilBERT
typically scores **15–25 percentage points higher** than the from-scratch
encoder under the same fine-tuning budget. With 20 k training examples and
3 epochs, the custom encoder usually reaches the 55–65% range while
DistilBERT lands around 80%+. The gap is the value of pre-training compressed
into one number.

**Where the custom encoder *wins*.** Trainable parameters (~1-2 M for our
tiny encoder vs ~67 M for DistilBERT), memory footprint, training time per
epoch, and full transparency about every weight in the model. If the task is
small and well-understood, a custom encoder can be a perfectly reasonable
production choice — cheaper to host and easier to audit.

**Attention behaviour.**
- **DistilBERT's last-layer attention** is highly structured. It tends to
  attend strongly to `[CLS]` and `[SEP]`, with secondary peaks on content
  words that link premise and hypothesis (matching nouns, contradicting
  verbs). Pre-training has given the model a clear notion of *which words
  matter for the task*.
- **The custom encoder's attention** is more diffuse — it has not yet
  learned to ignore stop words or to anchor on the special tokens. Patterns
  emerge but they are noisier. With more layers, more heads, and more
  training data, the picture would sharpen toward DistilBERT's.

**Trade-offs in one paragraph.** Custom multi-head attention stacks are an
excellent learning vehicle and a viable production choice for narrow tasks
where you have plenty of labeled data. For *general* NLU tasks like NLI,
fine-tuning a pretrained encoder is the strictly better default: the inductive
bias has been baked in during pre-training, and you pay for it once with
a few epochs of fine-tuning. The right question is rarely "DistilBERT or
custom?" — it is "how much pre-training does this task need, and what
model size can my deployment afford?"

**Insights about attention itself.**
- Self-attention has no built-in inductive bias toward locality — without
  positional encoding (Section 3.4 of the XP exercise), the model could not
  even distinguish word order.
- Multi-head attention lets the network represent several relational
  structures in parallel (syntactic, coreference, topical) inside the same
  layer — single-head attention squeezes all of those into one map.
- Residual connections + LayerNorm are what make stacking many encoder
  blocks practical; without them gradients die long before reaching the
  input embeddings.
